### Subspace model test

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".99"
import jax
import jax.numpy as jnp
from qwen_fineTuning_PtoC import QwenTextClassificationWrapper
import numpy as np
from flax.core import freeze, unfreeze
from qwen_fineTuning_PtoC_fixCps import Config, _setup_training_env, _prepare_datasets


%load_ext autoreload
%autoreload 2




In [4]:
# setup all
config = Config(data={
        'dataset_path': "artifacts/winogrande_m_dataset:v0/",
        'val_percentage': 0.1,
    })
config.net_kwargs['model_path'] = "artifacts/qwen2.5_7B_bfloat16:v0"
rng_key = jax.random.PRNGKey(0)
data, unique_target_ids, rng_key = _prepare_datasets(
        rng_key,
        dataset_path=config.data['dataset_path'],
        val_percentage=config.data.get('val_percentage', 0.0),
        batch_size=config.train_hyper.batch_size,
        smoke_test=config.train_hyper.smoke_test,
        logger=False)
config.net_kwargs['target_token_ids'] = unique_target_ids
env, params, rng_key = _setup_training_env(config, data, rng_key)
env

Dataset loaded: 2500 samples, 2 classes
Target token IDs: [[32]
 [33]]
Validation data are used
Training data: 2248 samples, Validation data: 252 samples
save best parameters


TrainingEnv(s_model=<src.subspace_curve.LoRACategorySubspace object at 0x7f36a0353e50>, optimizer=GradientTransformationExtraArgs(init=<function chain.<locals>.init_fn at 0x7f36687bb1c0>, update=<function chain.<locals>.update_fn at 0x7f36687bb250>), t_sample_fn=<function _setup_t_sample_fn.<locals>.t_sample_fn_v2 at 0x7f36a01168c0>, rho_scheduler=<function _setup_rho_scheduler.<locals>.<lambda> at 0x7f36a0159120>, valid_metrics_fn=<function _setup_validation.<locals>.valid_metrics_fn at 0x7f36687bab00>, empty_metric=({'val_loss': Array(inf, dtype=float32), 'val_acc': Array(-inf, dtype=float32), 'val_ece': Array(inf, dtype=float32), 'val_bma_ll': Array(-inf, dtype=float32), 'val_bma_acc': Array(-inf, dtype=float32), 'val_bma_ece': Array(inf, dtype=float32)}, None), get_best_params=<function _setup_validation.<locals>.get_best_params at 0x7f36687bab90>, train_step=<PjitFunction of <function _setup_training_env.<locals>._scan_jit at 0x7f36687bb6d0>>)

In [5]:
config.data

{'dataset_path': 'artifacts/winogrande_m_dataset:v0/', 'val_percentage': 0.1}

In [11]:
from jax import random
from qwen_fineTuning_PtoC_fixCps import _pad_to_batch, post_pred_performance, ece_fn
from functools import partial

def setup_metrics(model, batch_size, x, attention_mask, target, n_samples, use_linspace=False, average=True, key_prefix='', num_bins_ece=15, t_sample_fn=None, t_max=None, scan_data_loop=True):
    if t_max is None:
        t_max = getattr(model, 't_max', 1.0)

    # Fix 2: pad to full batches once at setup time — eliminates the
    # ``if len(idx_last) > 0`` Python branch inside acc_fn, which used to
    # produce arrays of different shapes and trigger JAX recompilation.
    x_pad, attn_pad, target_pad, n_valid = _pad_to_batch(x, attention_mask, target, batch_size)
    idx_ = jnp.arange(x_pad.shape[0]).reshape(-1, batch_size)  # always full batches

    # Build the t-sampling closure (n_s passed explicitly so get_t is shape-generic).
    if t_sample_fn is None:
        if use_linspace:
            def get_t_v1(key, n_s):
                # Return n_s evenly spaced samples in [0, t_max], excluding t_max itself.
                return jnp.linspace(0, t_max, n_s, endpoint=False)
            get_t = get_t_v1
        else:
            def get_t_v2(key, n_s):
                # Return n_s random samples in [0, t_max).
                return random.uniform(key, (n_s,), minval=0., maxval=t_max)
            get_t = get_t_v2
    else:
        if use_linspace:
            raise ValueError("Cannot use linspace sampling with custom t_sample_fn")
        def get_t_v3(key, n_s):
            return jax.vmap(lambda key: t_sample_fn(key))(random.split(key, n_s))
        get_t = get_t_v3

    # Fix 1 & 3: a single jitted evaluation core whose trace is stable as long
    # as the closed-over arrays (x_pad, attn_pad, target_pad) keep the same
    # shape.  ``n_s`` is declared static so JAX never conflates a 3-sample
    # smoke-test trace with the 20-sample full-eval trace.
    # @partial(jax.jit, static_argnames=('n_s',))
    def _eval_core(rng_key, params, n_s):
        def pred(rng_key, data_idx, _n_s=n_s):
            x_batch = x_pad.at[data_idx].get()
            att_m_batch = attn_pad.at[data_idx].get()
            # y_batch = target_pad.at[data_idx].get()

            rng_key, key_ = random.split(rng_key)
            t_tree = get_t(key_, _n_s)  # shape (n_s,)

            def single_pred(carry, t):
                key, x_b, att_b = carry
                key, subkey = random.split(key)
                out, _ = model(params['params'], {}, t, (x_b, att_b), train=False, key=subkey)
                return (key, x_b, att_b), out  # out shape (batch_size, output_dim)

            (rng_key, _, _), out = jax.lax.scan(single_pred, (rng_key, x_batch, att_m_batch), t_tree)
            return rng_key, out  # out shape (n_s, batch_size, output_dim)

        if scan_data_loop:
            # out: (n_batches, n_s, batch_size, output_dim)
            rng_key, out = jax.lax.scan(pred, rng_key, idx_)
        else:
            # jit_pred_fn = jax.jit(pred)
            out = []
            for data_idx in idx_:
                rng_key, out_batch = pred(rng_key, data_idx)  # (n_s, batch_size, output_dim)
                out.append(out_batch)
            out = jnp.stack(out, axis=0)  # (n_batches, n_s, batch_size, output_dim)
        # -> (n_s, n_padded, output_dim)  — trim padding
        out = out.transpose(1, 0, 2, 3).reshape(n_s, -1, out.shape[-1])[:, :n_valid, :]
        return rng_key, out  # (n_s, n_valid, output_dim)

    # Use shared post_pred_performance from src.utils (returns prefixed keys when key_prefix provided)

    def acc_fn(rng_key, params):
        rng_key, out = _eval_core(rng_key, params, n_samples)
        # out: (n_samples, n_valid, output_dim)

        bma_metrics = post_pred_performance(out, target, key_prefix=key_prefix + "bma_", num_bins=num_bins_ece)

        ece = jax.vmap(ece_fn, in_axes=(0, None, None))(out, target, num_bins_ece)  # (n_samples,)
        acc = jnp.argmax(out, axis=-1) == target  # (n_samples, n_valid)

        # Recompute NLL from returned logits — avoids accumulating a separate
        # loss tensor through the padded scan, eliminating shape mismatches.
        log_probs = jax.nn.log_softmax(out, axis=-1)  # (n_samples, n_valid, n_classes)
        loss = -jnp.sum(log_probs * jax.nn.one_hot(target, out.shape[-1]), axis=-1)  # (n_samples, n_valid)

        if average:
            ece = jnp.mean(ece)
            loss = loss.mean()
            acc = acc.sum() / (n_samples * n_valid)
        else:
            loss = jnp.mean(loss, axis=1)   # (n_samples,) — mean NLL per t-sample
            acc = jnp.sum(acc, axis=1) / n_valid

        return {key_prefix + 'loss': loss, key_prefix + 'acc': acc, key_prefix + 'ece': ece, **bma_metrics}, out
    return acc_fn

In [21]:
pred_fn = setup_metrics(env.s_model, 8, *data.get('val'), n_samples=2, use_linspace=True, key_prefix='test_', num_bins_ece=15, t_sample_fn=None, t_max=None, scan_data_loop=True)

In [22]:
pred_fn_jit = jax.jit(pred_fn)

In [23]:

metric, logits = pred_fn_jit(rng_key, params)

In [24]:
metric, logits = pred_fn_jit(rng_key, params)

In [10]:
pred_fn = setup_metrics(env.s_model, 32, *data.get('val'), n_samples=20, use_linspace=True, key_prefix='test_', num_bins_ece=15, t_sample_fn=None, t_max=None, scan_data_loop=False)

In [11]:
metric, logits = pred_fn(rng_key, params)

In [12]:
metric, logits = pred_fn(rng_key, params)

In [13]:
from jax import random
from qwen_fineTuning_PtoC_fixCps import _pad_to_batch, post_pred_performance, ece_fn
from functools import partial

@partial(jax.jit, static_argnames=('model',))
def _pred_jitted(model, rng_key, params, x_batch, att_m_batch, t_tree):
    def single_pred(key, t):
        key, subkey = random.split(key)
        out, _ = model(params['params'], {}, t, (x_batch, att_m_batch), train=False, key=subkey)
        return key, out
    rng_key, out = jax.lax.scan(single_pred, rng_key, t_tree)
    return rng_key, out

def setup_metrics(model, batch_size, x, attention_mask, target, n_samples, use_linspace=False, average=True, key_prefix='', num_bins_ece=15, t_sample_fn=None, t_max=None, scan_data_loop=True):
    if t_max is None:
        t_max = getattr(model, 't_max', 1.0)

    # Fix 2: pad to full batches once at setup time — eliminates the
    # ``if len(idx_last) > 0`` Python branch inside acc_fn, which used to
    # produce arrays of different shapes and trigger JAX recompilation.
    x_pad, attn_pad, target_pad, n_valid = _pad_to_batch(x, attention_mask, target, batch_size)
    idx_ = jnp.arange(x_pad.shape[0]).reshape(-1, batch_size)  # always full batches

    # Build the t-sampling closure (n_s passed explicitly so get_t is shape-generic).
    if t_sample_fn is None:
        if use_linspace:
            def get_t_v1(key, n_s):
                # Return n_s evenly spaced samples in [0, t_max], excluding t_max itself.
                return jnp.linspace(0, t_max, n_s, endpoint=False)
            get_t = get_t_v1
        else:
            def get_t_v2(key, n_s):
                # Return n_s random samples in [0, t_max).
                return random.uniform(key, (n_s,), minval=0., maxval=t_max)
            get_t = get_t_v2
    else:
        if use_linspace:
            raise ValueError("Cannot use linspace sampling with custom t_sample_fn")
        def get_t_v3(key, n_s):
            return jax.vmap(lambda key: t_sample_fn(key))(random.split(key, n_s))
        get_t = get_t_v3

    # Fix 1 & 3: a single jitted evaluation core whose trace is stable as long
    # as the closed-over arrays (x_pad, attn_pad, target_pad) keep the same
    # shape.  ``n_s`` is declared static so JAX never conflates a 3-sample
    # smoke-test trace with the 20-sample full-eval trace.
    # @partial(jax.jit, static_argnames=('n_s',))
    def _eval_core(rng_key, params, n_s):
        out = []
        rng_key, key_ = random.split(rng_key)
        t_tree = get_t(key_, n_samples)  # shape (n_s,)
        for data_idx in idx_:
            x_batch = x_pad[data_idx]
            att_m_batch = attn_pad[data_idx]
            rng_key, out_batch = _pred_jitted(model, rng_key, params, x_batch, att_m_batch, t_tree)
            out.append(out_batch)
        out = jnp.stack(out, axis=0)  # (n_batches, n_s, batch_size, output_dim)
        
        # -> (n_s, n_padded, output_dim)  — trim padding
        out = out.transpose(1, 0, 2, 3).reshape(n_s, -1, out.shape[-1])[:, :n_valid, :]
        return rng_key, out  # (n_s, n_valid, output_dim)

    # Use shared post_pred_performance from src.utils (returns prefixed keys when key_prefix provided)

    def acc_fn(rng_key, params):
        rng_key, out = _eval_core(rng_key, params, n_samples)
        # out: (n_samples, n_valid, output_dim)

        bma_metrics = post_pred_performance(out, target, key_prefix=key_prefix + "bma_", num_bins=num_bins_ece)

        ece = jax.vmap(ece_fn, in_axes=(0, None, None))(out, target, num_bins_ece)  # (n_samples,)
        acc = jnp.argmax(out, axis=-1) == target  # (n_samples, n_valid)

        # Recompute NLL from returned logits — avoids accumulating a separate
        # loss tensor through the padded scan, eliminating shape mismatches.
        log_probs = jax.nn.log_softmax(out, axis=-1)  # (n_samples, n_valid, n_classes)
        loss = -jnp.sum(log_probs * jax.nn.one_hot(target, out.shape[-1]), axis=-1)  # (n_samples, n_valid)

        if average:
            ece = jnp.mean(ece)
            loss = loss.mean()
            acc = acc.sum() / (n_samples * n_valid)
        else:
            loss = jnp.mean(loss, axis=1)   # (n_samples,) — mean NLL per t-sample
            acc = jnp.sum(acc, axis=1) / n_valid

        return {key_prefix + 'loss': loss, key_prefix + 'acc': acc, key_prefix + 'ece': ece, **bma_metrics}, out
    return acc_fn

In [ ]:
pred_fn = setup_metrics(env.s_model, 8, *data.get('val'), n_samples=20, use_linspace=True, key_prefix='test_', num_bins_ece=15, t_sample_fn=None, t_max=None, scan_data_loop=False)

In [15]:
metric, logits = pred_fn(rng_key, params)

In [24]:
pred_fn = setup_metrics(env.s_model, 8, *data.get('val'), n_samples=2, use_linspace=True, key_prefix='test_', num_bins_ece=15, t_sample_fn=None, t_max=None, scan_data_loop=False)

In [26]:
metric, logits = pred_fn(rng_key, params)

In [22]:
pred_fn = setup_metrics(env.s_model, 8, *data.get('test'), n_samples=20, use_linspace=True, key_prefix='test_', num_bins_ece=15, t_sample_fn=None, t_max=None, scan_data_loop=False)

In [23]:
metric, logits = pred_fn(rng_key, params)

In [27]:
from jax import random
from qwen_fineTuning_PtoC_fixCps import _pad_to_batch, post_pred_performance, ece_fn
from functools import partial

# def _pred_jitted(model, rng_key, params, x_batch, att_m_batch, t_tree):
@partial(jax.jit, static_argnames=('model',))
def single_pred(key, model, params, t, x_batch, att_m_batch):
    out, _ = model(params['params'], {}, t, (x_batch, att_m_batch), train=False, key=key)
    return out
    # rng_key, out = jax.lax.scan(single_pred, rng_key, t_tree)
    # return rng_key, out

def setup_metrics(model, batch_size, x, attention_mask, target, n_samples, use_linspace=False, average=True, key_prefix='', num_bins_ece=15, t_sample_fn=None, t_max=None, scan_data_loop=True):
    if t_max is None:
        t_max = getattr(model, 't_max', 1.0)

    # Fix 2: pad to full batches once at setup time — eliminates the
    # ``if len(idx_last) > 0`` Python branch inside acc_fn, which used to
    # produce arrays of different shapes and trigger JAX recompilation.
    x_pad, attn_pad, target_pad, n_valid = _pad_to_batch(x, attention_mask, target, batch_size)
    idx_ = jnp.arange(x_pad.shape[0]).reshape(-1, batch_size)  # always full batches

    # Build the t-sampling closure (n_s passed explicitly so get_t is shape-generic).
    if t_sample_fn is None:
        if use_linspace:
            def get_t_v1(key, n_s):
                # Return n_s evenly spaced samples in [0, t_max], excluding t_max itself.
                return jnp.linspace(0, t_max, n_s, endpoint=False)
            get_t = get_t_v1
        else:
            def get_t_v2(key, n_s):
                # Return n_s random samples in [0, t_max).
                return random.uniform(key, (n_s,), minval=0., maxval=t_max)
            get_t = get_t_v2
    else:
        if use_linspace:
            raise ValueError("Cannot use linspace sampling with custom t_sample_fn")
        def get_t_v3(key, n_s):
            return jax.vmap(lambda key: t_sample_fn(key))(random.split(key, n_s))
        get_t = get_t_v3

    # Fix 1 & 3: a single jitted evaluation core whose trace is stable as long
    # as the closed-over arrays (x_pad, attn_pad, target_pad) keep the same
    # shape.  ``n_s`` is declared static so JAX never conflates a 3-sample
    # smoke-test trace with the 20-sample full-eval trace.
    # @partial(jax.jit, static_argnames=('n_s',))
    def _eval_core(rng_key, params, n_s):
        out = []
        rng_key, key_ = random.split(rng_key)
        t_tree = get_t(key_, n_samples)  # shape (n_s,)
        for data_idx in idx_:
            x_batch = x_pad[data_idx]
            att_m_batch = attn_pad[data_idx]
            out_batch = []
            for t in t_tree:
                out_ = single_pred(rng_key, model, params, t, x_batch, att_m_batch)
                out_batch.append(out_)
            out.append(jnp.stack(out_batch, axis=0))  # (n_s, batch_size, output_dim)
                # rng_key, out_batch = _pred_jitted(model, rng_key, params, x_batch, att_m_batch, jnp.array([t]))
                # out.append(out_batch)
        out = jnp.stack(out, axis=0)  # (n_batches, n_s, batch_size, output_dim)
        
        # -> (n_s, n_padded, output_dim)  — trim padding
        out = out.transpose(1, 0, 2, 3).reshape(n_s, -1, out.shape[-1])[:, :n_valid, :]
        return rng_key, out  # (n_s, n_valid, output_dim)

    # Use shared post_pred_performance from src.utils (returns prefixed keys when key_prefix provided)

    def acc_fn(rng_key, params):
        rng_key, out = _eval_core(rng_key, params, n_samples)
        # out: (n_samples, n_valid, output_dim)

        bma_metrics = post_pred_performance(out, target, key_prefix=key_prefix + "bma_", num_bins=num_bins_ece)

        ece = jax.vmap(ece_fn, in_axes=(0, None, None))(out, target, num_bins_ece)  # (n_samples,)
        acc = jnp.argmax(out, axis=-1) == target  # (n_samples, n_valid)

        # Recompute NLL from returned logits — avoids accumulating a separate
        # loss tensor through the padded scan, eliminating shape mismatches.
        log_probs = jax.nn.log_softmax(out, axis=-1)  # (n_samples, n_valid, n_classes)
        loss = -jnp.sum(log_probs * jax.nn.one_hot(target, out.shape[-1]), axis=-1)  # (n_samples, n_valid)

        if average:
            ece = jnp.mean(ece)
            loss = loss.mean()
            acc = acc.sum() / (n_samples * n_valid)
        else:
            loss = jnp.mean(loss, axis=1)   # (n_samples,) — mean NLL per t-sample
            acc = jnp.sum(acc, axis=1) / n_valid

        return {key_prefix + 'loss': loss, key_prefix + 'acc': acc, key_prefix + 'ece': ece, **bma_metrics}, out
    return acc_fn

In [34]:
pred_fn = setup_metrics(env.s_model, 8, *data.get('test'), n_samples=2, use_linspace=True, key_prefix='test_', num_bins_ece=15, t_sample_fn=None, t_max=None, scan_data_loop=False)

In [35]:
metric, logits = pred_fn(rng_key, params)

In [30]:
metric, logits = pred_fn(rng_key, params)